In [ ]:
#!/usr/bin/env python
"""
Evaluate reward models on HH-RLHF TEST (Anthropic/hh-rlhf).

Requirements:
  pip install -q transformers datasets accelerate scikit-learn matplotlib pandas huggingface_hub tqdm
"""

import os
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import roc_auc_score
from google.colab import drive


# CONFIG

drive.mount("/content/drive")

HF_TOKEN = "xxxxxxxxxxxxxx"  #
os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN
os.environ["HF_TOKEN"] = HF_TOKEN

DATASET_NAME = "Anthropic/hh-rlhf"
EVAL_SPLIT = "test"

# ---- Models to compare (EDIT THESE REPO IDS) ----
MODEL_NO_AUG        = "xxxxxxxxxxxxxx_noaug"
MODEL_BASELINE_AUG  = "xxxxxxxxxxxxxx_uniaug"
MODEL_WON_RM        = "xxxxxxxxxxxxxx_WoN"
MODEL_OURS_AUG      = "xxxxxxxxxxxxxx_MARS"


MODELS = [
    ("No augmentation", MODEL_NO_AUG),
    ("Baseline (uniform aug.)", MODEL_BASELINE_AUG),
    ("WoN-trained RM", MODEL_WON_RM),
    ("MARS (adaptive aug.)", MODEL_OURS_AUG),

]

# Tokenization / eval
MAX_LENGTH = 512
BATCH_SIZE = 16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Eval protocol
NUM_ROUNDS = 3
SUBSAMPLE_SIZE = 100
BASE_SEED = 123

DO_BOOTSTRAP_CI = True
BOOTSTRAP_B = 300

OUT_DIR = "xxxxxxxxxxxxxx location to out DIR xxxxxxxxxxxxxx"
os.makedirs(OUT_DIR, exist_ok=True)

# ---- Bar colors ----
BAR_COLORS = {
    "No augmentation": "#59A14F",
    "Baseline (uniform aug.)": "#76B7B2",
    "MARS (adaptive aug.)": "#2F4B7C",
    "WoN-trained RM": "#E15759",
}
DEFAULT_BAR_COLOR = "#4C78A8"

# Matplotlib style helpers
def set_plot_style():
    plt.rcParams.update({
        "figure.dpi": 140,
        "savefig.dpi": 300,
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 12,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linestyle": "-",
        "legend.frameon": False,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
    })


def savefig(fig, path):
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    print(f"Saved: {path}")

def load_hh_rlhf_split():
    ds_dict = load_dataset(DATASET_NAME)
    if EVAL_SPLIT not in ds_dict:
        raise ValueError(f"Split '{EVAL_SPLIT}' not found. Available: {list(ds_dict.keys())}")
    ds = ds_dict[EVAL_SPLIT]

    needed = {"chosen", "rejected"}
    missing = needed - set(ds.column_names)
    if missing:
        raise ValueError(f"Split missing columns {missing}. Found: {ds.column_names}")
    return ds


def _ensure_pad_token(tok):
    if tok.pad_token is None:
        if tok.eos_token is not None:
            tok.pad_token = tok.eos_token
        else:
            tok.add_special_tokens({"pad_token": "[PAD]"})


def load_rm(model_name: str):
    """
    Loads RM as a sequence classifier and moves it to DEVICE.
    Works for typical reward models with a single regression head.
    """
    tok = AutoTokenizer.from_pretrained(model_name, use_fast=True, token=HF_TOKEN)
    _ensure_pad_token(tok)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, token=HF_TOKEN)
    if hasattr(model, "resize_token_embeddings"):
        model.resize_token_embeddings(len(tok))

    model.to(DEVICE)
    model.eval()
    return tok, model


def logits_to_scalar_scores(logits: np.ndarray) -> np.ndarray:
    logits = np.asarray(logits)
    if logits.ndim == 1:
        return logits.astype(np.float32)
    if logits.ndim == 2:
        if logits.shape[1] == 1:
            return logits[:, 0].astype(np.float32)
        if logits.shape[1] == 2:
            return logits[:, 1].astype(np.float32)
        return logits[:, -1].astype(np.float32)
    raise ValueError(f"Unexpected logits shape: {logits.shape}")


@torch.no_grad()
def score_texts(tokenizer, model, texts, batch_size=BATCH_SIZE):
    scores = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        out = model(**enc)
        logits = out.logits.detach().float().cpu().numpy()
        scores.append(logits_to_scalar_scores(logits))
    return np.concatenate(scores, axis=0)


def compute_metrics(chosen_scores, rejected_scores):
    chosen_scores = np.asarray(chosen_scores).reshape(-1)
    rejected_scores = np.asarray(rejected_scores).reshape(-1)
    assert chosen_scores.shape == rejected_scores.shape

    margins = chosen_scores - rejected_scores
    pairwise_acc = float(np.mean(margins > 0))

    y_true = np.concatenate([np.ones_like(chosen_scores), np.zeros_like(rejected_scores)])
    y_score = np.concatenate([chosen_scores, rejected_scores])
    auc = float(roc_auc_score(y_true, y_score))

    mean_margin = float(margins.mean())
    std_margin = float(margins.std())

    return {
        "pairwise_accuracy": pairwise_acc,
        "auc": auc,
        "mean_margin": mean_margin,
        "std_margin": std_margin,
        "margins": margins,
        "chosen_scores": chosen_scores,
        "rejected_scores": rejected_scores,
    }


def sample_indices(n, k, seed):
    rng = np.random.default_rng(seed)
    return [int(x) for x in rng.choice(n, size=k, replace=False)]


def bootstrap_ci_acc_auc(chosen_scores, rejected_scores, B=300, seed=0):
    """
    Bootstrap CIs over PAIRS (resample indices with replacement).
    Returns (acc_lo, acc_hi, auc_lo, auc_hi).
    """
    rng = np.random.default_rng(seed)
    cs = np.asarray(chosen_scores).reshape(-1)
    rs = np.asarray(rejected_scores).reshape(-1)
    n = cs.shape[0]
    idx_samples = rng.integers(0, n, size=(B, n))

    accs = np.empty(B, dtype=float)
    aucs = np.empty(B, dtype=float)

    for b in range(B):
        idx = idx_samples[b]
        c = cs[idx]
        r = rs[idx]
        accs[b] = float(np.mean((c - r) > 0))

        y_true = np.concatenate([np.ones_like(c), np.zeros_like(r)])
        y_score = np.concatenate([c, r])
        aucs[b] = float(roc_auc_score(y_true, y_score))

    acc_lo, acc_hi = np.quantile(accs, [0.025, 0.975])
    auc_lo, auc_hi = np.quantile(aucs, [0.025, 0.975])
    return float(acc_lo), float(acc_hi), float(auc_lo), float(auc_hi)

# Plotting
def _colors_for_tags(tags, bar_colors):
    if bar_colors is None:
        return [DEFAULT_BAR_COLOR for _ in tags]
    if isinstance(bar_colors, dict):
        return [bar_colors.get(t, DEFAULT_BAR_COLOR) for t in tags]
    if len(bar_colors) != len(tags):
        raise ValueError(f"bar_colors list must have length {len(tags)} but got {len(bar_colors)}")
    return list(bar_colors)


def plot_margin_distribution_pretty(tag, margins, mean_margin, std_margin, out_dir=OUT_DIR):
    margins = np.asarray(margins).reshape(-1)
    lo, hi = np.percentile(margins, [1, 99])
    if not np.isfinite(lo) or not np.isfinite(hi) or lo >= hi:
        lo, hi = float(margins.min()), float(margins.max())

    bins = 60
    density, bin_edges = np.histogram(margins, bins=bins, range=(lo, hi), density=True)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

    fig, ax = plt.subplots(figsize=(8, 6))
    line_color = BAR_COLORS.get(tag, "#1f77b4")

    ax.plot(bin_centers, density, linewidth=2.2, color=line_color)
    ax.fill_between(bin_centers, density, color=line_color, alpha=0.25)

    ax.set_title(f"{tag} — Margin distribution (HH-RLHF {EVAL_SPLIT})")
    ax.set_xlabel("Margin  r(chosen) − r(rejected)")
    ax.set_ylabel("Density")
    ax.grid(True, axis="y", alpha=0.25)

    txt = f"mean = {mean_margin:.4f}\nstd  = {std_margin:.4f}"
    ax.text(
        0.98, 0.98, txt,
        transform=ax.transAxes,
        ha="right", va="top",
        bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="none", alpha=0.95),
    )

    path = os.path.join(out_dir, f"{tag.replace(' ', '_').replace('/', '_')}_margin_density_line.png")
    savefig(fig, path)
    plt.show()


def plot_auc_acc_bars_pretty(mean_results, out_dir=OUT_DIR, bar_colors=None):
    tags = [r["tag"] for r in mean_results]
    auc_mean = np.array([r["auc_mean"] for r in mean_results], dtype=float)
    acc_mean = np.array([r["acc_mean"] for r in mean_results], dtype=float)

    x = np.arange(len(tags))
    colors = _colors_for_tags(tags, bar_colors)
    bar_width = 0.42

    # AUC
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.bar(x, auc_mean, width=bar_width, alpha=0.92, color=colors, edgecolor="black", linewidth=0.6)
    ax.set_ylabel("ROC-AUC (chosen=1, rejected=0)")
    ax.set_title(f"HH-RLHF {EVAL_SPLIT} — ROC-AUC")
    ax.set_xticks(x, labels=tags)
    ax.tick_params(axis="x", rotation=15)
    ax.set_ylim(0.45, 0.65)
    ax.grid(True, axis="y", alpha=0.25)
    for i, v in enumerate(auc_mean):
        ax.text(i, v + 0.005, f"{v:.3f}", ha="center", va="bottom", fontsize=10)
    savefig(fig, os.path.join(out_dir, f"hhrlhf_{EVAL_SPLIT}_auc_bar.png"))
    plt.show()

    # Accuracy
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.bar(x, acc_mean, width=bar_width, alpha=0.92, color=colors, edgecolor="black", linewidth=0.6)
    ax.set_ylabel("Pairwise Accuracy  P[r(chosen) > r(rejected)]")
    ax.set_title(f"HH-RLHF {EVAL_SPLIT} — Pairwise Accuracy")
    ax.set_xticks(x, labels=tags)
    ax.set_ylim(0.45, 0.65)
    ax.tick_params(axis="x", rotation=15)
    ax.grid(True, axis="y", alpha=0.25)
    for i, v in enumerate(acc_mean):
        ax.text(i, v + 0.005, f"{v:.3f}", ha="center", va="bottom", fontsize=10)
    savefig(fig, os.path.join(out_dir, f"hhrlhf_{EVAL_SPLIT}_acc_bar.png"))
    plt.show()


def plot_margin_trend(mean_results, out_dir, eval_split="test"):
    tags = [r["tag"] for r in mean_results]
    mean_margins = np.array([r["mean_margin_mean"] for r in mean_results], dtype=float)

    x = np.arange(len(tags))
    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    ax.plot(x, mean_margins, marker="o", linewidth=2.2, markersize=7)
    ax.set_xticks(x)
    ax.set_xticklabels(tags, rotation=15, ha="right")
    ax.set_ylabel("Mean margin  r(chosen) − r(rejected)")
    ax.set_title(f"Margin improvement across reward models (HH-RLHF {eval_split})")
    ax.grid(True, axis="y", alpha=0.25)
    for i, m in enumerate(mean_margins):
        ax.text(i, m + 0.01, f"{m:.3f}", ha="center", va="bottom", fontsize=10)
    savefig(fig, os.path.join(out_dir, f"hhrlhf_{eval_split}_margin_trend_line.png"))
    plt.show()


def _snr_from_mean_results(mean_results):
    mean_m = np.array([r["mean_margin_mean"] for r in mean_results], dtype=float)
    std_m = np.array([r["std_margin_mean"] for r in mean_results], dtype=float)
    eps = 1e-12
    return mean_m / np.maximum(std_m, eps)


def plot_snr_bars_pretty(mean_results, out_dir=OUT_DIR, bar_colors=None):
    tags = [r["tag"] for r in mean_results]
    snr = _snr_from_mean_results(mean_results)

    x = np.arange(len(tags))
    colors = _colors_for_tags(tags, bar_colors)
    bar_width = 0.42

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.bar(x, snr, width=bar_width, alpha=0.92, color=colors, edgecolor="black", linewidth=0.6)
    ax.set_ylabel("SNR = mean(margin) / std(margin)")
    ax.set_title(f"HH-RLHF {EVAL_SPLIT} — Margin SNR")
    ax.set_xticks(x, labels=tags)
    ax.tick_params(axis="x", rotation=15)
    ax.grid(True, axis="y", alpha=0.25)
    for i, v in enumerate(snr):
        ax.text(i, v + 0.01, f"{v:.3f}", ha="center", va="bottom", fontsize=10)
    savefig(fig, os.path.join(out_dir, f"hhrlhf_{EVAL_SPLIT}_snr_bar.png"))
    plt.show()


# Main
def main():
    set_plot_style()

    ds = load_hh_rlhf_split()
    n = len(ds)
    print(f"Loaded HH-RLHF '{EVAL_SPLIT}' split with {n} examples.")

    if SUBSAMPLE_SIZE is not None:
        print(f"Running {NUM_ROUNDS} rounds with subset size = {SUBSAMPLE_SIZE}")
    else:
        print(f"Running {NUM_ROUNDS} rounds on FULL dataset (may be slow).")

    per_round_rows = []
    metric_bank = {}
    agg_margins = {}
    loaded_models = []

    # Load models
    for tag, model_name in MODELS:
        print(f"\n=== Loading model: {tag} :: {model_name} ===")
        try:
            tok, model = load_rm(model_name)
        except Exception as e:
            print(f"[SKIP] Could not load {tag} ({model_name}). Reason: {repr(e)}")
            continue

        loaded_models.append((tag, model_name, tok, model))
        metric_bank[tag] = {
            "auc": [], "acc": [], "mean_margin": [], "std_margin": [],
            "acc_ci_lo": [], "acc_ci_hi": [], "auc_ci_lo": [], "auc_ci_hi": []
        }
        agg_margins[tag] = []

    if len(loaded_models) == 0:
        raise RuntimeError("No models loaded. Check HF_TOKEN and repo ids (private/gated?).")

    # Evaluate
    for r in range(NUM_ROUNDS):
        seed = BASE_SEED + r

        if SUBSAMPLE_SIZE is None:
            sub_ds = ds
            idx = None
        else:
            k = min(SUBSAMPLE_SIZE, n)
            idx = sample_indices(n, k, seed=seed)
            sub_ds = ds.select(idx)

        if idx is not None:
            idx_path = os.path.join(OUT_DIR, f"hhrlhf_{EVAL_SPLIT}_round{r+1}_n{k}_seed{seed}_indices.npy")
            np.save(idx_path, np.array(idx, dtype=np.int32))
            print(f"[Round {r+1}] Saved subset indices → {idx_path}")

        for tag, model_name, tok, model in loaded_models:
            chosen_scores = score_texts(tok, model, sub_ds["chosen"])
            rejected_scores = score_texts(tok, model, sub_ds["rejected"])
            m = compute_metrics(chosen_scores, rejected_scores)
            acc_lo = acc_hi = auc_lo = auc_hi = np.nan
            if DO_BOOTSTRAP_CI:
                acc_lo, acc_hi, auc_lo, auc_hi = bootstrap_ci_acc_auc(
                    chosen_scores, rejected_scores, B=BOOTSTRAP_B, seed=seed
                )

            print(
                f"{tag} | Round {r+1}/{NUM_ROUNDS}: "
                f"AUC={m['auc']:.4f} (CI {auc_lo:.4f}-{auc_hi:.4f}), "
                f"Acc={m['pairwise_accuracy']:.4f} (CI {acc_lo:.4f}-{acc_hi:.4f})"
            )

            per_round_rows.append({
                "round": r + 1,
                "seed": seed,
                "tag": tag,
                "model_name": model_name,
                "auc": m["auc"],
                "pairwise_accuracy": m["pairwise_accuracy"],
                "mean_margin": m["mean_margin"],
                "std_margin": m["std_margin"],
                "auc_ci_lo": auc_lo,
                "auc_ci_hi": auc_hi,
                "acc_ci_lo": acc_lo,
                "acc_ci_hi": acc_hi,
                "num_examples": len(sub_ds),
            })

            agg_margins[tag].append(m["margins"])
            metric_bank[tag]["auc"].append(m["auc"])
            metric_bank[tag]["acc"].append(m["pairwise_accuracy"])
            metric_bank[tag]["mean_margin"].append(m["mean_margin"])
            metric_bank[tag]["std_margin"].append(m["std_margin"])
            metric_bank[tag]["auc_ci_lo"].append(auc_lo)
            metric_bank[tag]["auc_ci_hi"].append(auc_hi)
            metric_bank[tag]["acc_ci_lo"].append(acc_lo)
            metric_bank[tag]["acc_ci_hi"].append(acc_hi)

    # Save per-round metrics
    per_round_df = pd.DataFrame(per_round_rows)
    per_round_path = os.path.join(OUT_DIR, f"hhrlhf_{EVAL_SPLIT}_metrics_per_round.csv")
    per_round_df.to_csv(per_round_path, index=False)
    print(f"\nSaved per-round metrics CSV: {per_round_path}")

    # Mean metrics per model
    mean_results = []
    for tag, model_name, _, _ in loaded_models:
        aucs = np.array(metric_bank[tag]["auc"], dtype=float)
        accs = np.array(metric_bank[tag]["acc"], dtype=float)
        mms = np.array(metric_bank[tag]["mean_margin"], dtype=float)
        sms = np.array(metric_bank[tag]["std_margin"], dtype=float)

        mean_results.append({
            "tag": tag,
            "model_name": model_name,
            "auc_mean": float(aucs.mean()),
            "acc_mean": float(accs.mean()),
            "mean_margin_mean": float(mms.mean()),
            "std_margin_mean": float(sms.mean()),
            "snr": float((mms.mean()) / max(sms.mean(), 1e-12)),
            # average CI endpoints across rounds
            "auc_ci_lo": float(np.nanmean(metric_bank[tag]["auc_ci_lo"])),
            "auc_ci_hi": float(np.nanmean(metric_bank[tag]["auc_ci_hi"])),
            "acc_ci_lo": float(np.nanmean(metric_bank[tag]["acc_ci_lo"])),
            "acc_ci_hi": float(np.nanmean(metric_bank[tag]["acc_ci_hi"])),
        })

    mean_df = pd.DataFrame(mean_results)

    # Save ranked summary
    rank_df = mean_df.sort_values(["auc_mean", "acc_mean"], ascending=False).reset_index(drop=True)
    rank_path = os.path.join(OUT_DIR, f"hhrlhf_{EVAL_SPLIT}_summary_ranked.csv")
    rank_df.to_csv(rank_path, index=False)
    print(f"Saved ranked summary CSV: {rank_path}")

    mean_path = os.path.join(OUT_DIR, f"hhrlhf_{EVAL_SPLIT}_metrics_mean.csv")
    mean_df.to_csv(mean_path, index=False)
    print(f"Saved mean metrics CSV: {mean_path}")

    # Per-model margin density plots
    for tag, _, _, _ in loaded_models:
        margins_all = np.concatenate(agg_margins[tag], axis=0)
        mm = float(np.mean(metric_bank[tag]["mean_margin"]))
        sm = float(np.mean(metric_bank[tag]["std_margin"]))
        plot_margin_distribution_pretty(tag, margins_all, mean_margin=mm, std_margin=sm, out_dir=OUT_DIR)

    # AUC/Acc bars
    mean_results_for_plot = [
        {"tag": r["tag"], "auc_mean": r["auc_mean"], "acc_mean": r["acc_mean"]}
        for r in mean_results
    ]
    plot_auc_acc_bars_pretty(
        mean_results_for_plot,
        out_dir=OUT_DIR,
        bar_colors=BAR_COLORS,
    )

    # Margin trend, SNR bars
    plot_margin_trend(mean_results, out_dir=OUT_DIR, eval_split=EVAL_SPLIT)
    plot_snr_bars_pretty(mean_results, out_dir=OUT_DIR, bar_colors=BAR_COLORS)

    print("\nDone. Outputs saved to:")
    print(OUT_DIR)
    print("\nTopline summary (sorted by AUC then Acc):")
    display_cols = ["tag", "auc_mean", "auc_ci_lo", "auc_ci_hi", "acc_mean", "acc_ci_lo", "acc_ci_hi", "snr"]
    print(rank_df[display_cols].to_string(index=False))


if __name__ == "__main__":
    main()
